# NL2SQL-FR — Fine-tuning QLoRA d'un LLM open source

Affinage de **Qwen2.5-Coder-1.5B-Instruct** pour traduire des questions métier en français en SQL, sur un SI commercial télécom **fictif**.
Évaluation par **exactitude d'exécution** sur des formulations jamais vues à l'entraînement, avant et après fine-tuning.

**Pour lancer :** Google Colab → *Exécution › Modifier le type d'exécution › GPU T4*, puis *Exécution › Tout exécuter*.
Durée indicative sur T4 : 30 à 45 minutes au total. Fonctionne aussi sur Kaggle (GPU T4 ou P100).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -U "transformers>=4.46" "peft>=0.13" "trl>=0.20" "bitsandbytes>=0.44" accelerate datasets

In [ ]:
import os, sys
if not os.path.exists("nl2sql-fr-qlora"):
    !git clone -q https://github.com/rooniweb/nl2sql-fr-qlora
%cd nl2sql-fr-qlora
sys.path.insert(0, os.getcwd())
!python scripts/preparer_donnees.py

## 1. Chargement du modèle en 4 bits

In [ ]:
import inspect, json, time
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from nl2sql import jeu
from nl2sql.evaluation import evaluer
from nl2sql.prompts import exemple_sft, extraire_sql, messages_prompt

MODELE = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16
BASE = "data/telecom_fictif.db"
os.makedirs("resultats", exist_ok=True)

tok = AutoTokenizer.from_pretrained(MODELE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
model = AutoModelForCausalLM.from_pretrained(MODELE, quantization_config=bnb, device_map="auto", torch_dtype=DTYPE)
train = jeu.lire_jsonl("data/train.jsonl")
test = jeu.lire_jsonl("data/test.jsonl")
print(f"Train : {len(train)} · Test : {len(test)}")

In [ ]:
def generer_sql(model, questions, lot=8, max_new_tokens=200):
    tok.padding_side = "left"
    model.eval()
    sorties = []
    for i in range(0, len(questions), lot):
        textes = [tok.apply_chat_template(messages_prompt(q), tokenize=False, add_generation_prompt=True) for q in questions[i:i + lot]]
        enc = tok(textes, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        for j in range(len(textes)):
            sorties.append(extraire_sql(tok.decode(out[j, enc["input_ids"].shape[1]:], skip_special_tokens=True)))
    return sorties

def evaluer_modele(model, nom):
    t0 = time.time()
    preds = generer_sql(model, [e["question"] for e in test])
    duree = time.time() - t0
    jeu.ecrire_jsonl([{"id": e["id"], "question": e["question"], "sql": p} for e, p in zip(test, preds)], f"resultats/predictions_{nom}.jsonl")
    r = evaluer(BASE, test, {e["id"]: p for e, p in zip(test, preds)})
    r["secondes_par_question"] = round(duree / len(test), 2)
    print(f"{nom} : exactitude d'exécution {r['exactitude']} % ({r['justes']}/{r['n']}), erreurs SQL {r['erreurs_sql']}, {r['secondes_par_question']} s/question")
    return r

## 2. Référence : le modèle de base, sans fine-tuning

In [ ]:
res_base = evaluer_modele(model, 'base')

## 3. Fine-tuning QLoRA (PEFT + TRL)

La perte ne porte que sur la requête SQL (format *prompt / completion*), pas sur le schéma ni la question.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
ds = Dataset.from_list([exemple_sft(e) for e in train])

params = dict(output_dir="sorties/qlora", num_train_epochs=2, per_device_train_batch_size=8, gradient_accumulation_steps=2,
              learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03, logging_steps=10, save_strategy="no",
              bf16=BF16, fp16=not BF16, gradient_checkpointing=True, max_length=1024, max_seq_length=1024,
              completion_only_loss=True, report_to="none", seed=42)
acceptes = set(inspect.signature(SFTConfig.__init__).parameters)
config = SFTConfig(**{k: v for k, v in params.items() if k in acceptes})      # compatible avec plusieurs versions de TRL
cle_tok = "processing_class" if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters else "tokenizer"

tok.padding_side = "right"
trainer = SFTTrainer(model=model, args=config, train_dataset=ds, peft_config=lora, **{cle_tok: tok})
t0 = time.time()
stats = trainer.train()
duree_entrainement = round((time.time() - t0) / 60, 1)
trainer.save_model("sorties/adaptateur_lora")
print(f"Entraînement : {duree_entrainement} min · perte finale {stats.training_loss:.4f}")
trainer.model.print_trainable_parameters()

## 4. Évaluation du modèle affiné

In [ ]:
res_ft = evaluer_modele(trainer.model, 'qlora')

In [ ]:
resultats = {
    "modele": MODELE, "methode": "QLoRA (4 bits nf4, LoRA r=16, alpha=32, 2 époques)",
    "gpu": torch.cuda.get_device_name(0), "train": len(train), "test": len(test),
    "duree_entrainement_min": duree_entrainement, "perte_finale": round(stats.training_loss, 4),
    "base": {k: res_base[k] for k in ("exactitude", "justes", "n", "erreurs_sql", "secondes_par_question", "par_gabarit")},
    "qlora": {k: res_ft[k] for k in ("exactitude", "justes", "n", "erreurs_sql", "secondes_par_question", "par_gabarit")},
}
json.dump(resultats, open("resultats/resultats.json", "w"), ensure_ascii=False, indent=2)
print(f"| Modèle | Exactitude d'exécution | Erreurs SQL |\n|---|---|---|")
print(f"| {MODELE} (base) | {res_base['exactitude']} % | {res_base['erreurs_sql']} |")
print(f"| + QLoRA | {res_ft['exactitude']} % | {res_ft['erreurs_sql']} |")

## 5. Récupérer les résultats

Télécharge `resultats/resultats.json` (et les deux fichiers de prédictions) puis ajoute-les au dépôt, ou envoie-les pour mettre à jour le README, le CV et le portfolio.
Optionnel : publier l'adaptateur LoRA sur Hugging Face avec `trainer.model.push_to_hub("<ton-compte>/nl2sql-fr-qlora")` après `huggingface-cli login`.

In [ ]:
try:
    from google.colab import files
    for f in ["resultats/resultats.json", "resultats/predictions_base.jsonl", "resultats/predictions_qlora.jsonl"]:
        files.download(f)
except ImportError:
    print("Hors Colab : les fichiers sont dans le dossier resultats/.")